# Discogs XML → Parquet
Reads the `.xml.gz` dumps, saves each schema as JSON, and writes Parquet.
The first run on a dump infers schemas; later runs reuse them and skip the inference pass.

## 1. Settings
Edit these paths. Set `DATE` to the date in the file names.

In [8]:
import os, json, time
from pathlib import Path

DUMP_DIR   = Path("/mnt/data/public/discogs/2026").expanduser()          # folder with the .xml.gz files
DATE       = "20261101"                              # e.g. "20200101" for the old dump
OUT_DIR    = Path("discogs/parquet")
SCHEMA_DIR = Path("discogs/schemas")

SAMPLING_RATIO = 0.05   # only used when inferring
FORCE_INFER    = False  # True = ignore saved schemas and infer again

ROW_TAGS = {"artists": "artist", "labels": "label",
            "masters": "master", "releases": "release"}

SCHEMA_DIR.mkdir(parents=True, exist_ok=True)

## 2. Start Spark


In [11]:
import pyspark
from pyspark.sql import SparkSession

spark = (SparkSession
    .builder
    .master('local[*]')
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate())

## 3. Helpers

In [12]:
def schema_path(name):
    return SCHEMA_DIR / f"{name}_schema.json"

def load_schema(name):
    p = schema_path(name)
    if FORCE_INFER or not p.exists():
        return None
    return StructType.fromJson(json.loads(p.read_text()))

def save_schema(name, schema):
    schema_path(name).write_text(json.dumps(json.loads(schema.json()), indent=2))

def read_xml(name):
    """Read one dump file, using a saved schema if there is one."""
    src = DUMP_DIR / f"discogs_{DATE}_{name}.xml.gz"
    reader = spark.read.format("xml").option("rowTag", ROW_TAGS[name])
    schema = load_schema(name)
    t0 = time.time()
    if schema is not None:
        print(f"[{name}] using saved schema")
        df = reader.schema(schema).load(str(src))
    else:
        print(f"[{name}] inferring schema ...")
        df = reader.option("samplingRatio", SAMPLING_RATIO).load(str(src))
        save_schema(name, df.schema)
        print(f"[{name}] schema saved ({time.time()-t0:.0f}s)")
    return df

def to_parquet(name, df):
    dst = OUT_DIR / DATE / name
    t0 = time.time()
    df.repartition(PARTITIONS).write.mode("overwrite").parquet(str(dst))
    out = spark.read.parquet(str(dst))
    print(f"[{name}] wrote {out.count():,} rows, {len(out.columns)} cols in {time.time()-t0:.0f}s")

## 4. Try a small file first
Artists is the quickest way to check everything works before the long releases run.

In [13]:
artists = read_xml("artists")
artists.printSchema()

[artists] inferring schema ...
[artists] schema saved (445s)
root
 |-- aliases: struct (nullable = true)
 |    |-- name: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _VALUE: string (nullable = true)
 |    |    |    |-- _id: long (nullable = true)
 |-- data_quality: string (nullable = true)
 |-- groups: struct (nullable = true)
 |    |-- name: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _VALUE: string (nullable = true)
 |    |    |    |-- _id: long (nullable = true)
 |-- id: long (nullable = true)
 |-- members: struct (nullable = true)
 |    |-- name: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _VALUE: string (nullable = true)
 |    |    |    |-- _id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- namevariations: struct (nullable = true)
 |    |-- name: array (nullable = true)
 |    |    |-- element: string (contai

In [15]:
artists.select('aliases.name').show(5, truncate=60)

+------------------------------------------------------------+
|                                                        name|
+------------------------------------------------------------+
|[{Jesper Dahlbäck, 239}, {Groove Machine, 16055}, {Dick T...|
|[{Puente Latino, 2470}, {Yakari & Delano, 19536}, {Crushe...|
|[{Size 9, 11217}, {The Crusher, 95949}, {Dinky Dog, 28405...|
|[{The Trinity, 12490}, {Unity Gain, 23236}, {Jim Henson P...|
|[{Goldwave, 15673}, {Apathism, 22282}, {Toko, 39918}, {X-...|
+------------------------------------------------------------+
only showing top 5 rows


In [16]:
to_parquet("artists", artists)

[artists] wrote 10,284,281 rows, 10 cols in 208s


## 5. Convert the rest
Releases is the slow one: each `.gz` file is decompressed on a single core.

In [ ]:
for name in ["labels", "masters", "releases"]:
    df = read_xml(name)
    to_parquet(name, df)

## 6. Load from Parquet for analysis
From here on, read only the Parquet copies; they're fast and parallel.

In [ ]:
releases = spark.read.parquet(str(OUT_DIR / DATE / "releases"))
artists  = spark.read.parquet(str(OUT_DIR / DATE / "artists"))
labels   = spark.read.parquet(str(OUT_DIR / DATE / "labels"))
masters  = spark.read.parquet(str(OUT_DIR / DATE / "masters"))

releases.select("_id", "title", "country", "released").show(5)